# 02. 지도 학습을 이용한 회귀(1) — 선형 회귀 실습
> 교재 29~41p · SK네트웍스 Family AI 캠프 37기 · 2026-10-14

**회귀(Regression)**: 학습 데이터의 특성과 상관 관계를 파악해, 처음 보는 데이터의 값을 **연속적인 숫자**로 예측하는 것

**선형 회귀(Linear Regression)**: 입력과 출력의 관계를 직선 `y = Wx + b`로 나타내고, 데이터 분포를 **가장 잘 표현하는 직선**을 찾아 예측

**다중 선형 회귀**: 특성(feature)을 여러 개 사용 → `y = w₁x₁ + w₂x₂ + … + wₚxₚ + b`

In [1]:
# 필요한 라이브러리 임포트
import numpy as np
import pandas as pd

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

## 1. 개념 확인 — 공부 시간 vs 시험 성적
| x (hour) | y (score) |
|---|---|
| 9 | 90 |
| 8 | 80 |
| 4 | 40 |
| 2 | 20 |

👉 **7시간 공부하면 성적은 몇 점일까?**

In [2]:
from sklearn.linear_model import LinearRegression

# scikit-learn의 X는 반드시 2차원(행: 샘플, 열: 특성)
X_hour = np.array([[9], [8], [4], [2]])
y_score = np.array([90, 80, 40, 20])

model = LinearRegression()
model.fit(X_hour, y_score)

print(f'기울기 W: {model.coef_[0]:.2f}')        # 10.00
print(f'절편   b: {round(model.intercept_, 2) + 0.0:.2f}')       # 0.00 (-0.00 표시 방지)
print(f'7시간 공부 시 예상 성적: {model.predict([[7]])[0]:.1f}점')   # 70.0

기울기 W: 10.00
절편   b: 0.00
7시간 공부 시 예상 성적: 70.0점


## 2. 회귀 모델의 성능 평가 지표
오차(Error, loss)가 **작을수록** 좋은 모델이다.

| 지표 | 수식 | 특징 |
|---|---|---|
| **MAE** (평균 절대값 오차) | $\frac{\sum\|y-\hat{y}\|}{n}$ | 오차의 절대값 평균, 이상치에 덜 민감 |
| **MSE** (평균 제곱 오차) | $\frac{\sum(y-\hat{y})^2}{n}$ | 큰 오차에 더 큰 벌점, 단위가 제곱 |
| **RMSE** (평균 제곱근 오차) | $\sqrt{MSE}$ | MSE에 루트 → **정답과 같은 단위**로 해석 가능 |

In [3]:
# 직접 계산해 보고, sklearn 함수 결과와 비교
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

y_true = np.array([3.0, 5.0, 2.5, 7.0])
y_pred = np.array([2.5, 5.0, 4.0, 8.0])

mae = np.mean(np.abs(y_true - y_pred))
mse = np.mean((y_true - y_pred) ** 2)
rmse_val = np.sqrt(mse)

print(f'MAE : 직접 {mae:.4f} | sklearn {mean_absolute_error(y_true, y_pred):.4f}')
print(f'MSE : 직접 {mse:.4f} | sklearn {mean_squared_error(y_true, y_pred):.4f}')
print(f'RMSE: 직접 {rmse_val:.4f} | sklearn {root_mean_squared_error(y_true, y_pred):.4f}')

MAE : 직접 0.7500 | sklearn 0.7500
MSE : 직접 0.8750 | sklearn 0.8750
RMSE: 직접 0.9354 | sklearn 0.9354


## 3. 보스턴 주택 가격 예측 (LinearRegression)
- 데이터: 집값에 영향을 주는 특성 **13가지**, **506개**
- 정답(target): `medv` — 주택 가격 중앙값 (단위: $1,000)
- 목표: LinearRegression 모델 **학습 → 예측 → 평가**

| 컬럼 | 의미 | 컬럼 | 의미 |
|---|---|---|---|
| crim | 1인당 범죄 발생 수 | dis | 보스턴 고용시설 5곳까지 거리 |
| zn | 25,000평방피트 이상 주거구역 비중 | rad | 순환고속도로 접근성 |
| indus | 비소매 상업지역 면적 비율 | tax | $10,000당 재산세율 |
| chas | 찰스강 인접 여부(1: 인접) | ptratio | 학생/교사 비율 |
| nox | 일산화질소 농도 | b | 지역별 흑인 비율 지표 |
| rm | 평균 방 개수 | lstat | 저소득층 인구 비율(%) |
| age | 1940년 이전 건축 비율 | **medv** | **주택 가격(정답)** |

### 3-1. 데이터 불러오기

In [4]:
# 노트북과 같은 폴더에 house_price.csv를 두고 실행
# (다른 위치라면 경로만 수정: 예) r'C:\Users\내이름\work\house_price.csv')
file_path = './house_price.csv'

df = pd.read_csv(file_path)
print(f'데이터 크기: {df.shape}')   # (506, 14)
display(df.head())

데이터 크기: (506, 14)


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


### 3-2. 데이터 전처리 — 누락 데이터 확인

In [5]:
# isnull(): 누락 여부(True/False) -> sum(): True 개수 합산
num_nulls = df.isnull().sum()
print(f'전체 누락 데이터 수: {num_nulls.sum()}')
print(num_nulls)

전체 누락 데이터 수: 0
crim       0
zn         0
indus      0
chas       0
nox        0
rm         0
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64


### 3-3. 학습용 / 평가용 데이터 생성 (75 : 25)

In [6]:
from sklearn.model_selection import train_test_split

# X: 정답 컬럼(medv)을 제외한 13개 특성 / y: 정답 컬럼
X_data = df.drop(columns=['medv'])
y_data = df['medv']

# random_state 고정 -> 실행할 때마다 같은 분할 (결과 재현 가능)
# 회귀 문제는 정답이 연속값이라 stratify를 사용하지 않는다
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data,
    test_size=0.25,
    random_state=0
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')   # (379, 13), (127, 13)

# X와 y의 행 인덱스가 같은 순서로 섞였는지 확인
print('X_train / y_train 인덱스 일치:', (X_train.index == y_train.index).all())

X_train: (379, 13), X_test: (127, 13)
X_train / y_train 인덱스 일치: True


In [7]:
# 요약 통계량: 컬럼마다 평균/표준편차(스케일)가 크게 다름 -> 표준화 필요
display(X_train.describe().T[['mean', 'std', 'min', 'max']])

,mean,std,min,max
crim,3.412736,8.258225,0.00632,88.9762
zn,11.861478,23.810122,0.00000,100.0000
indus,11.083509,7.021496,0.46000,27.7400
chas,0.065963,0.248546,0.00000,1.0000
nox,0.556160,0.116179,0.38500,0.8710
rm,6.312683,0.693391,3.56100,8.7800
age,68.735620,28.214667,2.90000,100.0000
dis,3.781929,2.094931,1.17420,12.1265
rad,9.411609,8.676092,1.00000,24.0000
tax,403.226913,171.319579,187.00000,711.0000


### 3-4. 표준화(StandardScaler)
- 각 특성을 **평균 0, 표준편차 1**로 변환 → 스케일 차이로 인한 왜곡을 줄임
- ⚠️ **학습용은 `fit_transform()`**, **평가용은 `transform()`만**
  (평가용 데이터의 정보가 학습에 새어 들어가는 **데이터 누수**를 막기 위함)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습용: 학습용 데이터의 평균/표준편차를 계산(fit)하고 적용(transform)
X_train_scaled = scaler.fit_transform(X_train)
# 평가용: 학습용에서 구한 평균/표준편차를 그대로 적용
X_test_scaled = scaler.transform(X_test)

print('표준화 후 학습용 평균(≈0):', X_train_scaled.mean(axis=0).round(2))
print('표준화 후 학습용 표준편차(≈1):', X_train_scaled.std(axis=0).round(2))

표준화 후 학습용 평균(≈0): [ 0.  0.  0.  0. -0.  0.  0. -0.  0.  0.  0. -0.  0.]
표준화 후 학습용 표준편차(≈1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### 3-5. LinearRegression 모델 생성 · 학습

In [9]:
from sklearn.linear_model import LinearRegression

# 모델 생성
lireg = LinearRegression()

# 학습: 학습용 데이터(379개)에 대해 오차(MSE)가 최소인 직선(초평면)을 찾음
lireg.fit(X_train_scaled, y_train)
print('학습 완료!')

학습 완료!


In [10]:
# 학습 결과: 특성별 기울기(가중치)와 절편
coef_lr = pd.Series(lireg.coef_, index=X_data.columns, name='LinearRegression')
print(f'절편: {lireg.intercept_:.4f}')
display(coef_lr.sort_values().to_frame())
# 표준화했기 때문에 기울기 크기를 '집값에 미치는 영향력'으로 비교할 수 있다
# 예) rm(+): 방이 많을수록 비쌈 / lstat(-): 저소득층 비율이 높을수록 쌈

절편: 22.6087


,LinearRegression
lstat,-3.590270
dis,-3.002166
ptratio,-2.157438
tax,-1.932890
nox,-1.808769
crim,-0.971001
age,-0.198233
indus,-0.040448
chas,0.594088
b,0.751991


### 3-6. 평가용 데이터로 예측

In [11]:
pred_test = lireg.predict(X_test_scaled)

# 정답과 예측을 나란히 비교
compare = pd.DataFrame({'정답': y_test.values, '예측': pred_test.round(2)}, index=y_test.index)
compare['오차'] = (compare['정답'] - compare['예측']).round(2)
display(compare.head(10))

,정답,예측,오차
329,22.6,24.95,-2.35
371,50.0,23.62,26.38
219,23.0,29.21,-6.21
403,8.3,11.96,-3.66
78,21.2,21.33,-0.13
15,19.9,19.47,0.43
487,20.6,20.42,0.18
340,18.7,21.52,-2.82
310,16.1,18.99,-2.89
102,18.6,19.95,-1.35


### 3-7. 모델 평가 (MAE / MSE / RMSE)

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import root_mean_squared_error as rmse


def regression_report(name, y_true, y_pred):
    """회귀 평가지표 3종을 딕셔너리로 반환"""
    return {
        'model': name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
    }


rmse_test = rmse(y_test, pred_test)
print(f'평가용 데이터에 대한 성능(RMSE): {rmse_test:.4f}')   # 약 5.46 -> 평균적으로 약 $5,460 오차

평가용 데이터에 대한 성능(RMSE): 5.4573


## 4. 규제(Regularization) — Lasso (L1)
- 기울기(가중치)에 **벌점**을 줘서 모델을 단순하게 만들어 **과대적합을 방지**
- Lasso(L1)는 영향이 작은 특성의 기울기를 **0으로 만들어 특성 선택** 효과가 있다
- `alpha`가 클수록 규제가 강해진다 (기본값 1.0)

In [13]:
from sklearn.linear_model import Lasso

# 모델 생성 · 학습
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)

# 기울기 비교: Lasso는 일부 특성의 기울기가 0이 된다
coef_cmp = pd.concat([coef_lr, pd.Series(lasso.coef_, index=X_data.columns, name='Lasso')], axis=1)
display(coef_cmp.round(3))
print('Lasso가 0으로 만든 특성:', list(coef_cmp.index[coef_cmp['Lasso'] == 0]))

,LinearRegression,Lasso
crim,-0.971,-0.000
zn,1.047,0.000
indus,-0.040,-0.000
chas,0.594,0.000
nox,-1.809,-0.000
rm,2.610,2.529
age,-0.198,-0.000
dis,-3.002,-0.000
rad,2.080,-0.000
tax,-1.933,-0.228


Lasso가 0으로 만든 특성: ['crim', 'zn', 'indus', 'chas', 'nox', 'age', 'dis', 'rad']


In [14]:
# 예측 · 평가
pred_test_lasso = lasso.predict(X_test_scaled)
rmse_lasso = rmse(y_test, pred_test_lasso)
print(f'평가용 데이터에 대한 성능(RMSE): {rmse_lasso:.4f}')

평가용 데이터에 대한 성능(RMSE): 5.9573


## 5. 결과 비교

In [15]:
result = pd.DataFrame([
    regression_report('LinearRegression', y_test, pred_test),
    regression_report('Lasso(alpha=1.0)', y_test, pred_test_lasso),
]).set_index('model')

display(result.round(4))

,MAE,MSE,RMSE
model,,,
LinearRegression,3.6683,29.7822,5.4573
Lasso(alpha=1.0),3.9285,35.4896,5.9573


## ✅ 정리
- 회귀 = 연속값 예측 / 선형 회귀 = 오차가 최소인 직선 `y = Wx + b` 찾기
- 평가지표 **MAE · MSE · RMSE** — 작을수록 좋음, RMSE는 정답과 단위가 같아 해석이 쉽다
- 표준화는 **학습용 `fit_transform` / 평가용 `transform`** (데이터 누수 방지)
- Lasso(L1)는 불필요한 특성의 기울기를 0으로 → 모델 단순화.
  이 데이터에서는 규제가 성능을 약간 낮췄으므로 `alpha`를 줄여(예: 0.1) 다시 비교해 볼 것